# 黄金内外盘价差分析工具

用于分析伦敦金价与中国AU9999价格之间的价差

## 计算公式

**公式1：换算AU9999价格**
$$AU9999 = \frac{伦敦金价格 \times 汇率}{金衡盎司}$$

其中：金衡盎司 = 31.1035克

**公式2：内外盘价差**
$$价差 = 换算AU9999 - 实际AU9999$$

- 价差 > 0：国内价格偏低 🟢
- 价差 < 0：国内价格偏高 🔴

In [ ]:
# ==================== 常量定义 ====================
ORE_TROY_OUNCE = 31.1035  # 金衡盎司 = 31.1035克

# 颜色定义（ANSI 转义码）
class Colors:
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BLUE = '\033[94m'
    BOLD = '\033[1m'
    RESET = '\033[0m'

# 价差阈值（用于高亮提示）
THRESHOLD_LOW = 0.5   # 价差比 < 0.5% 时国内偏高
THRESHOLD_HIGH = 1.0  # 价差比 > 1.0% 时国内偏低明显

print("✅ 常量加载完成")

## 🔧 工具函数定义

包含以下功能模块：
- **数据验证**：检查数据有效性
- **汇率获取**：多来源获取 USD/CNY 汇率
- **价格获取**：获取黄金价格数据
- **错误处理**：友好的错误提示和重试机制

In [ ]:
# ==================== 导入依赖 ====================
import akshare as ak
import pandas as pd
import math
import requests
import re
from datetime import datetime

# 禁用 tqdm 进度条（避免 IProgress 错误）
try:
    from tqdm import tqdm
    tqdm.disable = True
except ImportError:
    pass

# ==================== 数据验证工具 ====================
def is_valid_number(x):
    """检查是否为有效的数字（非 None、非 NaN、非零）"""
    if x is None:
        return False
    try:
        val = float(x)
        return not math.isnan(val) and val > 0
    except (TypeError, ValueError):
        return False

def format_timestamp(dt=None):
    """格式化时间戳"""
    if dt is None:
        dt = datetime.now()
    return dt.strftime('%Y-%m-%d %H:%M:%S')

print("✅ 基础工具加载完成")

In [ ]:
# ==================== 汇率获取模块 ====================
def fetch_exchange_rate_sina():
    """从新浪财经获取 USD/CNY 汇率"""
    try:
        url = 'https://hq.sinajs.cn/list=USDCNY'
        headers = {'Referer': 'https://finance.sina.com.cn'}
        resp = requests.get(url, headers=headers, timeout=10)
        resp.encoding = 'gbk'
        text = resp.text
        match = re.search(r'var hq_str_USDCNY="([^"]+)"', text)
        if match:
            parts = match.group(1).split(',')
            if len(parts) >= 7:
                rate = float(parts[6]) if parts[6] else float(parts[1])
                if is_valid_number(rate):
                    return rate, "新浪财经"
    except requests.Timeout:
        return None, "新浪财经(超时)"
    except Exception as e:
        return None, f"新浪财经(错误)"
    return None, None

def fetch_exchange_rate_exchangerate_api():
    """从 exchangerate-api 获取 USD/CNY 汇率"""
    try:
        url = 'https://api.exchangerate-api.com/v4/latest/USD'
        resp = requests.get(url, timeout=10)
        data = resp.json()
        rate = data.get('rates', {}).get('CNY')
        if is_valid_number(rate):
            return rate, "ExchangeRate-API"
    except requests.Timeout:
        return None, "ExchangeRate-API(超时)"
    except Exception as e:
        return None, f"ExchangeRate-API(错误)"
    return None, None

def fetch_exchange_rate_akshare():
    """从 akshare 获取 USD/CNY 汇率"""
    try:
        fx_df = ak.fx_spot_quote()
        # 尝试中文列名
        if '货币对' in fx_df.columns and '买报价' in fx_df.columns:
            usd_cny_row = fx_df[fx_df['货币对'] == 'USD/CNY']
            if not usd_cny_row.empty:
                raw_value = usd_cny_row.iloc[0]['买报价']
                if is_valid_number(raw_value):
                    return float(raw_value), "akshare"
        # 尝试英文列名
        if 'code' in fx_df.columns and 'bid' in fx_df.columns:
            usd_cny_row = fx_df[fx_df['code'] == 'USD/CNY']
            if not usd_cny_row.empty:
                raw_value = usd_cny_row.iloc[0]['bid']
                if is_valid_number(raw_value):
                    return float(raw_value), "akshare"
    except Exception as e:
        return None, f"akshare(错误)"
    return None, None

print("✅ 汇率获取模块加载完成")

### 汇率获取（多源备选）

优先级：新浪财经 > ExchangeRate-API > akshare

In [ ]:
# ==================== 多源汇率获取 ====================
def fetch_exchange_rate_multi_source():
    """
    多来源获取 USD/CNY 汇率
    返回: (汇率值, 来源名称, 所有源状态)
    """
    sources = [
        ("新浪财经", fetch_exchange_rate_sina),
        ("ExchangeRate-API", fetch_exchange_rate_exchangerate_api),
        ("akshare", fetch_exchange_rate_akshare),
    ]
    
    status_list = []  # 记录各源状态
    
    for source_name, fetch_func in sources:
        rate, actual_source = fetch_func()
        if rate is not None:
            status_list.append((source_name, "✅", rate))
            return rate, actual_source, status_list
        else:
            status_list.append((source_name, "❌", actual_source or "失败"))
    
    return None, None, status_list

print("✅ 多源汇率获取函数定义完成")

## 📡 数据获取

自动获取实时价格数据（从 akshare）：
- **外盘**：COMEX 黄金期货（美元/盎司）
- **汇率**：美元/人民币外汇实时行情
- **内盘**：AU9999.SGE 上海黄金交易所现货（元/克）

In [ ]:
# ==================== 价格获取模块 ====================
def fetch_gold_prices():
    """
    获取黄金价格数据
    
    返回: dict {
        'london_price': 外盘价格,
        'exchange_rate': 汇率,
        'actual_au9999': 内盘价格,
        'fetch_time': 获取时间,
        'sources': 各数据源,
        'status': 各数据获取状态
    }
    """
    result = {
        'london_price': None,
        'exchange_rate': None,
        'actual_au9999': None,
        'fetch_time': format_timestamp(),
        'sources': {},
        'status': {'au9999': None, 'rate': [], 'comex': None}
    }
    
    print(f"{Colors.BLUE}⏳ 开始获取数据...{Colors.RESET}")
    print(f"   获取时间: {result['fetch_time']}")
    print("-" * 50)
    
    # 1. 获取 AU9999 价格
    print("【1/3】获取 AU9999 价格...")
    try:
        sge_df = ak.spot_hist_sge(symbol="Au99.99")
        if not sge_df.empty and 'close' in sge_df.columns:
            raw_value = sge_df.iloc[-1]['close']
            if is_valid_number(raw_value):
                result['actual_au9999'] = float(raw_value)
                result['sources']['au9999'] = '上海黄金交易所'
                result['status']['au9999'] = '✅'
                print(f"   {Colors.GREEN}✅ AU9999: ¥{result['actual_au9999']:.2f}/克{Colors.RESET}")
            else:
                result['status']['au9999'] = '❌ 数据无效'
                print(f"   {Colors.RED}❌ AU9999: 数据无效（空值或NaN）{Colors.RESET}")
        else:
            result['status']['au9999'] = '❌ 格式异常'
            print(f"   {Colors.RED}❌ AU9999: 数据格式异常{Colors.RESET}")
    except Exception as e:
        result['status']['au9999'] = f'❌ {str(e)[:20]}'
        print(f"   {Colors.RED}❌ AU9999 获取失败: {e}{Colors.RESET}")
    
    # 2. 获取汇率
    print("\n【2/3】获取汇率...")
    rate, source, status_list = fetch_exchange_rate_multi_source()
    result['status']['rate'] = status_list
    
    if rate is not None:
        result['exchange_rate'] = rate
        result['sources']['rate'] = source
        print(f"   {Colors.GREEN}✅ 汇率: {rate:.4f} (来源: {source}){Colors.RESET}")
    else:
        print(f"   {Colors.RED}❌ 所有汇率源均获取失败{Colors.RESET}")
        for src, status, val in status_list:
            print(f"      - {src}: {status}")
    
    # 3. 获取 COMEX 黄金期货价格
    print("\n【3/3】获取 COMEX 黄金期货价格...")
    try:
        futures_df = ak.futures_global_spot_em()
        gold_mask = futures_df['名称'].str.contains('COMEX黄金|GC', case=False, na=False)
        if gold_mask.any():
            gold_futures = futures_df[gold_mask].copy()
            if '成交量' in gold_futures.columns:
                gold_futures['成交量_num'] = pd.to_numeric(gold_futures['成交量'], errors='coerce').fillna(0)
                main_contract = gold_futures.loc[gold_futures['成交量_num'].idxmax()]
            else:
                main_contract = gold_futures.iloc[0]

            if '最新价' in main_contract and pd.notna(main_contract['最新价']):
                raw_value = main_contract['最新价']
                if is_valid_number(raw_value):
                    result['london_price'] = float(raw_value)
                    result['sources']['comex'] = main_contract['名称']
                    result['status']['comex'] = '✅'
                    print(f"   {Colors.GREEN}✅ COMEX黄金: ${result['london_price']:.2f}/盎司{Colors.RESET}")
                    print(f"      合约: {main_contract['名称']}")
                else:
                    result['status']['comex'] = '❌ 数据无效'
                    print(f"   {Colors.RED}❌ COMEX: 价格数据无效{Colors.RESET}")
            else:
                result['status']['comex'] = '❌ 无最新价'
                print(f"   {Colors.RED}❌ COMEX: 无最新价数据{Colors.RESET}")
        else:
            result['status']['comex'] = '❌ 未找到'
            print(f"   {Colors.RED}❌ 未找到 COMEX 黄金期货数据{Colors.RESET}")
    except Exception as e:
        result['status']['comex'] = f'❌ {str(e)[:20]}'
        print(f"   {Colors.RED}❌ COMEX 获取失败: {e}{Colors.RESET}")
    
    print("-" * 50)
    return result

print("✅ 价格获取模块加载完成")

### 执行数据获取

In [ ]:
# ==================== 执行数据获取 ====================
# 默认值（当实时获取失败时使用）
DEFAULT_VALUES = {
    'london_price': 2990.00,      # COMEX 黄金期货（美元/盎司）
    'exchange_rate': 7.25,        # 美元/人民币汇率
    'actual_au9999': 690.00,      # AU9999 价格（元/克）
}

# 获取实时数据
fetch_result = fetch_gold_prices()

# 确定最终使用的值（实时数据优先，失败则使用默认值）
london_price = fetch_result['london_price'] if is_valid_number(fetch_result['london_price']) else DEFAULT_VALUES['london_price']
exchange_rate = fetch_result['exchange_rate'] if is_valid_number(fetch_result['exchange_rate']) else DEFAULT_VALUES['exchange_rate']
actual_au9999 = fetch_result['actual_au9999'] if is_valid_number(fetch_result['actual_au9999']) else DEFAULT_VALUES['actual_au9999']
fetch_time = fetch_result['fetch_time']

# 统计获取状态
success_count = sum([
    is_valid_number(fetch_result['london_price']),
    is_valid_number(fetch_result['exchange_rate']),
    is_valid_number(fetch_result['actual_au9999'])
])

# 显示汇总
print(f"\n{Colors.BOLD}{'='*50}{Colors.RESET}")
print(f"{Colors.BOLD}📊 数据获取汇总{Colors.RESET}")
print(f"{Colors.BOLD}{'='*50}{Colors.RESET}")
print(f"   数据时间: {fetch_time}")
print(f"   成功获取: {success_count}/3 项")

# 显示最终使用的数据
print(f"\n   {Colors.BOLD}当前使用的数据:{Colors.RESET}")
if is_valid_number(fetch_result['london_price']):
    print(f"   ├─ 外盘(COMEX): ${london_price:.2f}/盎司 {Colors.GREEN}✅ 实时{Colors.RESET}")
else:
    print(f"   ├─ 外盘(COMEX): ${london_price:.2f}/盎司 {Colors.YELLOW}⚠️ 默认值{Colors.RESET}")

if is_valid_number(fetch_result['exchange_rate']):
    print(f"   ├─ 汇率: {exchange_rate:.4f} {Colors.GREEN}✅ 实时{Colors.RESET}")
else:
    print(f"   ├─ 汇率: {exchange_rate:.4f} {Colors.YELLOW}⚠️ 默认值{Colors.RESET}")

if is_valid_number(fetch_result['actual_au9999']):
    print(f"   └─ 内盘(AU9999): ¥{actual_au9999:.2f}/克 {Colors.GREEN}✅ 实时{Colors.RESET}")
else:
    print(f"   └─ 内盘(AU9999): ¥{actual_au9999:.2f}/克 {Colors.YELLOW}⚠️ 默认值{Colors.RESET}")

print(f"{Colors.BOLD}{'='*50}{Colors.RESET}")

## 📈 计算结果

基于上述数据进行价差计算和分析

In [ ]:
# ==================== 计算与结果展示 ====================

def highlight_value(value, threshold_low=0.5, threshold_high=1.0, is_ratio=True):
    """
    根据值的大小返回高亮颜色
    - 正值且大：绿色（国内偏低，买入机会）
    - 负值或小：红色（国内偏高）
    - 中等：黄色
    """
    if is_ratio:
        if value >= threshold_high:
            return Colors.GREEN  # 国内明显偏低
        elif value <= -threshold_low:
            return Colors.RED    # 国内明显偏高
        else:
            return Colors.YELLOW  # 适中
    return Colors.RESET

def get_market_advice(diff_ratio):
    """根据价差比给出投资建议"""
    if diff_ratio >= THRESHOLD_HIGH:
        return f"{Colors.GREEN}💡 国内价格明显偏低，可考虑买入{Colors.RESET}"
    elif diff_ratio >= 0:
        return f"{Colors.YELLOW}💡 内外盘价格基本平衡{Colors.RESET}"
    elif diff_ratio >= -THRESHOLD_LOW:
        return f"{Colors.YELLOW}💡 国内价格略高，建议观望{Colors.RESET}"
    else:
        return f"{Colors.RED}💡 国内价格明显偏高，不建议买入{Colors.RESET}"

# ========== 核心计算 ==========
converted_au9999 = london_price * exchange_rate / ORE_TROY_OUNCE
price_diff = converted_au9999 - actual_au9999
diff_ratio = price_diff / actual_au9999 * 100

# 获取高亮颜色
highlight_color = highlight_value(diff_ratio)
advice = get_market_advice(diff_ratio)

# ========== 结果展示 ==========
print(f"\n{Colors.BOLD}{'='*60}{Colors.RESET}")
print(f"{Colors.BOLD}                    📊 价差分析结果{Colors.RESET}")
print(f"{Colors.BOLD}{'='*60}{Colors.RESET}")
print(f"   📅 数据时间: {Colors.BLUE}{fetch_time}{Colors.RESET}")

print(f"\n{Colors.BOLD}【输入数据】{Colors.RESET}")
print(f"   ├─ 外盘(COMEX): ${london_price:.2f}/盎司")
print(f"   ├─ 汇率: {exchange_rate:.4f}")
print(f"   └─ 内盘(AU9999): ¥{actual_au9999:.2f}/克")

print(f"\n{Colors.BOLD}【公式1】换算AU9999价格{Colors.RESET}")
print(f"   {london_price:.2f} × {exchange_rate:.4f} ÷ {ORE_TROY_OUNCE}")
print(f"   = {Colors.BLUE}{converted_au9999:.2f}{Colors.RESET} 元/克")

print(f"\n{Colors.BOLD}【公式2】内外盘价差{Colors.RESET}")
print(f"   {converted_au9999:.2f} - {actual_au9999:.2f}")
print(f"   = {highlight_color}{price_diff:+.2f}{Colors.RESET} 元/克")

if price_diff > 0:
    print(f"   → {Colors.GREEN}国内价格偏低{Colors.RESET}（换算后外盘更贵）")
elif price_diff < 0:
    print(f"   → {Colors.RED}国内价格偏高{Colors.RESET}（换算后外盘更便宜）")
else:
    print(f"   → 内外盘价格一致")

print(f"\n{Colors.BOLD}【公式3】内外盘价差比{Colors.RESET} {Colors.BOLD}(关键指标){Colors.RESET}")
print(f"   {price_diff:+.2f} ÷ {actual_au9999:.2f} × 100%")
print(f"   = {highlight_color}{Colors.BOLD}{diff_ratio:+.2f}%{Colors.RESET}")

# ========== 阈值说明 ==========
print(f"\n{Colors.BOLD}【阈值参考】{Colors.RESET}")
print(f"   ├─ 价差比 > +{THRESHOLD_HIGH}%: {Colors.GREEN}国内明显偏低 🟢{Colors.RESET}")
print(f"   ├─ 价差比 ±{THRESHOLD_LOW}%: {Colors.YELLOW}基本平衡 🟡{Colors.RESET}")
print(f"   └─ 价差比 < -{THRESHOLD_LOW}%: {Colors.RED}国内明显偏高 🔴{Colors.RESET}")

# ========== 投资建议 ==========
print(f"\n{Colors.BOLD}{'─'*60}{Colors.RESET}")
print(f"{Colors.BOLD}【投资建议】{Colors.RESET}")
print(f"   {advice}")
print(f"{Colors.BOLD}{'='*60}{Colors.RESET}")

## 📋 批量场景计算

输入多组数据批量计算价差：

In [ ]:
# ==================== 批量场景计算 ====================

def highlight_dataframe_cell(val, column_name):
    """为 DataFrame 单元格添加颜色高亮"""
    if column_name in ['价差', '价差比']:
        if isinstance(val, str):
            val_num = float(val.replace('%', '').replace('+', ''))
            if val_num >= THRESHOLD_HIGH:
                return f'color: green; font-weight: bold'
            elif val_num <= -THRESHOLD_LOW:
                return f'color: red; font-weight: bold'
            else:
                return f'color: orange'
    return ''

# 批量计算：2026年1-3月真实历史数据
# 日期：原始日期 → 顺延后工作日
# 2026-01-01(周四,元旦) → 01-02(周五)
# 2026-02-01(周日) → 02-03(周一)
# 2026-03-01(周日) → 03-02(周一)
scenarios = [
    (4350, 7.03, 980.00, "2026-01-02"),   # 1月2日（元旦顺延）
    (4913, 6.95, 1060.00, "2026-02-03"),  # 2月3日（周末顺延）
    (5088, 6.88, 1170.00, "2026-03-02"),  # 3月2日（周末顺延）
]

# 计算并生成表格数据
table_data = []
for london, rate, au9999, desc in scenarios:
    converted = london * rate / ORE_TROY_OUNCE
    diff = converted - au9999
    ratio = diff / au9999 * 100
    
    # 根据价差比添加 emoji
    if ratio >= THRESHOLD_HIGH:
        emoji = '🟢'
    elif ratio <= -THRESHOLD_LOW:
        emoji = '🔴'
    else:
        emoji = '🟡'
    
    table_data.append([f"{desc} {emoji}", london, rate, au9999, converted, diff, ratio])

# 创建 DataFrame
df_scenarios = pd.DataFrame(table_data, columns=['日期', '外盘($)', '汇率', 'AU9999', '换算价', '价差', '价差比(%)'])

# 格式化数值列
df_scenarios['外盘($)'] = df_scenarios['外盘($)'].map('{:.2f}'.format)
df_scenarios['汇率'] = df_scenarios['汇率'].map('{:.4f}'.format)
df_scenarios['AU9999'] = df_scenarios['AU9999'].map('{:.2f}'.format)
df_scenarios['换算价'] = df_scenarios['换算价'].map('{:.2f}'.format)
df_scenarios['价差'] = df_scenarios['价差'].map('{:+.2f}'.format)
df_scenarios['价差比(%)'] = df_scenarios['价差比(%)'].map('{:+.2f}%'.format)

# 应用样式函数 - 只返回 CSS 样式，不修改内容
def color_cells(styler):
    """为 DataFrame 添加颜色样式"""
    styles = pd.DataFrame('', index=styler.index, columns=styler.columns)
    
    for i in styler.index:
        ratio_str = styler.loc[i, '价差比(%)']
        ratio_val = float(ratio_str.replace('%', '').replace('+', ''))
        
        if ratio_val >= THRESHOLD_HIGH:
            color = 'green'
        elif ratio_val <= -THRESHOLD_LOW:
            color = 'red'
        else:
            color = 'orange'
        
        styles.loc[i, '价差'] = f'color: {color}; font-weight: bold;'
        styles.loc[i, '价差比(%)'] = f'color: {color}; font-weight: bold;'
    
    return styles

# 显示表格
print(f"\n{Colors.BOLD}2026年1-3月首日历史数据价差分析{Colors.RESET}")
print(f"{Colors.YELLOW}注：非工作日已顺延至下一工作日{Colors.RESET}\n")

styled_df = df_scenarios.style.apply(color_cells, axis=None).set_table_styles([
    {'selector': 'th', 'props': 'background-color: #f0f0f0; font-weight: bold; border: 1px solid #ccc;'},
    {'selector': 'td', 'props': 'border: 1px solid #ccc; padding: 5px;'},
]).hide(axis='index')

display(styled_df)

# 显示图例
print(f"\n{Colors.BOLD}图例说明:{Colors.RESET}")
print(f"  🟢 价差比 > +{THRESHOLD_HIGH}%: 国内偏低")
print(f"  🟡 价差比 ±{THRESHOLD_LOW}%: 基本平衡")
print(f"  🔴 价差比 < -{THRESHOLD_LOW}%: 国内偏高")

# 数据来源说明
print(f"\n{Colors.BOLD}数据来源:{Colors.RESET}")
print(f"  - COMEX: MarketWatch, Fortune")
print(f"  - 汇率: CFETS (中国外汇交易中心)")
print(f"  - AU9999: 上海黄金交易所")